In [1]:
import os
from dotenv import load_dotenv
load_dotenv() ## Loading all the environement variable

groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
## LLM Model
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)

c:\Users\User\anaconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A875FE5AE0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A875FE61D0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is Shivansh")])

AIMessage(content="Nice to meet you, Shivansh. I'm an AI assistant, here to help with any questions or topics you'd like to discuss. How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 43, 'total_tokens': 81, 'completion_time': 0.154411147, 'completion_tokens_details': None, 'prompt_time': 0.002260316, 'prompt_tokens_details': None, 'queue_time': 0.052874304, 'total_time': 0.156671463}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019da5d4-4a4c-76b0-94c6-eee794353e34-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 38, 'total_tokens': 81})

In [5]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, My name is Shivansh"),
        AIMessage(content="Nice to meet you, Shivansh. Is there something I can help you with or would you like to chat?"),
        HumanMessage(content="Hey what's my name and what do I do?")

    ]
)

AIMessage(content="Your name is Shivansh. As for what you do, I don't have any prior information about you. We just started our conversation, so I don't know anything about your profession, hobbies, or interests. Would you like to share something about yourself?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 88, 'total_tokens': 142, 'completion_time': 0.094860203, 'completion_tokens_details': None, 'prompt_time': 0.008040135, 'prompt_tokens_details': None, 'queue_time': 0.059958794, 'total_time': 0.102900338}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019da5d4-4e36-7113-ad5f-ebdc35a76df6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 88, 'output_tokens': 54, 'total_tokens': 142})

In [6]:
## Message History
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [7]:
config={"configurable": {"session_id":"chat1"}}

In [8]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Shivansh")],
    config=config
)

In [9]:
response.content

'Nice to meet you, Shivansh. Is there something I can help you with or would you like to chat?'

In [10]:
with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Shivansh")],
    config=config
)

AIMessage(content="Nice to meet you again, Shivansh. We just had a brief conversation, so I'm happy to chat with you some more. What's on your mind today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 85, 'total_tokens': 121, 'completion_time': 0.054719316, 'completion_tokens_details': None, 'prompt_time': 0.005342088, 'prompt_tokens_details': None, 'queue_time': 0.101301162, 'total_time': 0.060061404}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019da5d4-5268-7d50-91fe-7ece60b3434a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 'output_tokens': 36, 'total_tokens': 121})

In [11]:
## Change the config  --> session id
config1={"configurable":{"session_id":"chat2"}}
response = with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config = config1
)
response.content

"I don't have any information about your name. This is the beginning of our conversation, and I don't retain information about individual users. If you'd like to share your name, I can address you by that name for the duration of our conversation."

## Prompt Templates

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer all the question the next of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | model

In [13]:
chain.invoke({"messages":[HumanMessage(content="Hi I am shivansh")]})

AIMessage(content="Nice to meet you, Shivansh. I'm here to help with any questions or topics you'd like to discuss. How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 57, 'total_tokens': 91, 'completion_time': 0.037599576, 'completion_tokens_details': None, 'prompt_time': 0.004259408, 'prompt_tokens_details': None, 'queue_time': 0.052437567, 'total_time': 0.041858984}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019da5d4-5705-70c3-9faf-51ba3d2bc0bc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 34, 'total_tokens': 91})

In [14]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [15]:
config = {"configurable":{"session_id":"chats3"}}
response = with_message_history.invoke(
    [HumanMessage(content="Hi My name is Shivansh")],
    config = config
)

response.content

"Nice to meet you, Shivansh. I'm happy to assist you with any questions or information you may need. How's your day going so far?"

In [16]:
## Add multiple input response
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [17]:
response = chain.invoke({"messages":[HumanMessage(content="Hi My name is shivansh")],"language":"Hindi"})
response.content

'नमस्ते शिवांश! मैं आपकी सहायता करने के लिए तैयार हूँ। क्या आपके पास कोई प्रश्न या चिंता है, जिस पर मैं आपकी सहायता कर सकता हूँ?'

## Lets now wrap this more complicated chain in a message history class. This time,because there are multiple keys in the inpt, we need to specify the correct key to use to save the chat hisory.

In [18]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [19]:
config = {"configurable":{"session_id":"chat4"}}
response = with_message_history.invoke(
     {"messages": [HumanMessage(content="Hi, I am Shivansh")],"language":"Hindi"},
     config = config
)

response.content

'नमस्ते शिवांश! मैं आपकी सहायता करने के लिए तैयार हूँ। कृपया अपना प्रश्न पूछें या बात करने के लिए तैयार रहें।'

In [20]:
response = with_message_history.invoke(
     {"messages": [HumanMessage(content="What is my name")],"language":"Hindi"},
     config = config
)

response.content

'आपका नाम शिवांश है।'

## Manage the Conversation History

In [21]:
from langchain_core.messages import SystemMessage,trim_messages

In [22]:
trimmer = trim_messages(
    max_tokens=270,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    
)
messages = [
    SystemMessage(content="you are a good assistant"),
    HumanMessage(content="Hi! I'm a bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="Whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="Yes!"),
]
trimmer.invoke(messages)

c:\Users\User\anaconda3\envs\myenv\lib\site-packages\langchain_core\language_models\base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


[SystemMessage(content='you are a good assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="Hi! I'm a bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Yes!', additional_kwa

In [23]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer) | prompt | model  
)

chain.invoke(
    {"messages":messages + [HumanMessage(content = "What ice cream do i Like")],
    "language":"English"}
)

response.content

'आपका नाम शिवांश है।'

In [28]:
## Lets wrap this is in the Message History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chats5"}}

In [29]:
response = with_message_history.invoke(
    {
        "messages":messages+[HumanMessage(content="Whats my name?")],
        "language":"English",
    },
    config=config,
)
response.content

'I remember, your name is Bob!'